# ANEXO E. Semana 3: transfer learning con imágenes

Notebook del curso SENA Iniciación a la IA para ADSO 2026. Las explicaciones completas están en la guía, anexo E.

Guarde su copia con **Archivo > Guardar una copia en Drive**.

Use GPU: **Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4**.

## E.1 Instalar librerías

In [ ]:
!pip install -q -U transformers datasets accelerate

Puede aparecer un mensaje rojo que empieza con `ERROR: pip's dependency resolver...` sobre conflictos con otras librerías de Colab (por ejemplo `cudf` o `pyarrow`). Si la celda termina con el check verde ✓, ignórelo: no afecta el curso.

## E.2 Cargar las imágenes

Conjunto de hojas de fríjol (1.295 imágenes): sana, mancha angular y roya.

In [ ]:
from datasets import load_dataset

ds = load_dataset("AI-Lab-Makerere/beans")
etiquetas = ds["train"].features["labels"].names
print(ds)
print(etiquetas)
ds["train"][0]["image"]

## E.3 Ajustar el modelo

In [ ]:
import numpy as np
import torch
from transformers import (AutoImageProcessor, AutoModelForImageClassification,
                          TrainingArguments, Trainer)

base = "google/vit-base-patch16-224-in21k"
procesador = AutoImageProcessor.from_pretrained(base)


def transformar(lote):
    entradas = procesador([img.convert("RGB") for img in lote["image"]], return_tensors="pt")
    entradas["labels"] = lote["labels"]
    return entradas


ds_listo = ds.with_transform(transformar)


def juntar(lote):
    return {"pixel_values": torch.stack([x["pixel_values"] for x in lote]),
            "labels": torch.tensor([x["labels"] for x in lote])}


def metricas(p):
    return {"exactitud": (np.argmax(p.predictions, axis=1) == p.label_ids).mean()}


modelo = AutoModelForImageClassification.from_pretrained(
    base, num_labels=len(etiquetas),
    id2label=dict(enumerate(etiquetas)),
    label2id={e: i for i, e in enumerate(etiquetas)})

args = TrainingArguments(
    output_dir="vit-frijol",
    per_device_train_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    eval_strategy="epoch",
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none",
)

entrenador = Trainer(model=modelo, args=args,
                     train_dataset=ds_listo["train"],
                     eval_dataset=ds_listo["validation"],
                     data_collator=juntar, compute_metrics=metricas)
entrenador.train()

Con GPU T4 el entrenamiento tarda alrededor de un minuto (más la descarga del modelo, de 346 MB). Observe cómo baja la pérdida (loss) y sube la exactitud en cada época: en la prueba de esta guía pasó de 94,7 % a 100 % en validación.

Al cargar el modelo aparece un reporte con las palabras UNEXPECTED y MISSING: es normal, significa que se reemplazó la última capa del modelo para que clasifique las 3 clases del fríjol.

## E.4 Evaluar y probar

In [ ]:
print(entrenador.evaluate(ds_listo["test"]))

In [ ]:
from transformers import pipeline

clasificador = pipeline("image-classification", model=entrenador.model,
                        image_processor=procesador, device=0)
clasificador(ds["test"][5]["image"])

Para probar con una foto suya: ícono de carpeta de la barra izquierda > subir archivo, y luego `clasificador("nombre_de_su_foto.jpg")`.

## E.5 Publicar en Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
entrenador.model.push_to_hub("vit-frijol")
procesador.push_to_hub("vit-frijol")

El modelo queda en `https://huggingface.co/SU_USUARIO/vit-frijol`. Allí haga clic en Edit model card y escriba la ficha con la plantilla del Anexo H.2.